# Module 06 -- Monte Carlo Pricing

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

Monte Carlo is the Swiss army knife of derivatives pricing. When you can't find
a closed-form solution -- and for most interesting products, you can't -- you
simulate. The idea is simple: generate thousands of possible future price paths,
compute the payoff on each one, and average. Law of large numbers does the rest.

On the desk, MC is what we use for path-dependent exotics: barriers, Asians,
lookbacks, autocallables. For vanillas you'd never use MC (BSM is instant and exact),
but it's the perfect teaching tool because you can verify MC against the closed form.

*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


## Geometric Brownian Motion (GBM)

Under the risk-neutral measure, the stock follows:

$$S_T = S_0 \exp\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right]$$

where $Z \sim N(0,1)$. For multi-step paths (needed for barriers), we discretize:

$$S_{t+dt} = S_t \exp\left[\left(r - \frac{\sigma^2}{2}\right)dt + \sigma\sqrt{dt}\,Z_t\right]$$


In [ ]:
def simulate_gbm_terminal(S0, r, sigma, T, n_paths, seed=42):
    """Simulate terminal values only (for European options)."""
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_paths)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    return ST

def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, seed=42):
    """Simulate full paths (for path-dependent options)."""
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    Z = rng.standard_normal((n_steps, n_paths))

    S = np.zeros((n_steps + 1, n_paths))
    S[0] = S0
    for t in range(n_steps):
        S[t + 1] = S[t] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[t])
    return S


## Visualizing GBM Paths

Let's plot some paths to build intuition. Notice how they fan out over time --
that's the diffusion component. The drift is barely visible because for typical
parameters, the randomness dominates.


In [ ]:
S0, r, sigma, T = 100, 0.05, 0.20, 1.0
n_steps = 252  # daily steps for 1 year

paths = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths=50, seed=123)
t_axis = np.linspace(0, T, n_steps + 1)

fig, ax = plt.subplots(figsize=(12, 5))
for i in range(50):
    ax.plot(t_axis, paths[:, i], lw=0.6, alpha=0.5)
ax.axhline(S0, color='black', ls='--', lw=1, label=f'S0 = {S0}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Spot Price')
ax.set_title('50 GBM Paths -- S0=100, vol=20%, r=5%')
ax.legend()
plt.tight_layout()
plt.show()


## Pricing a European Call -- MC vs. BSM

The whole point of this exercise: simulate terminal prices, compute payoffs,
discount, average. Then check against BSM.


In [ ]:
K = 100

def bsm_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# BSM benchmark
bsm_price = bsm_call_price(S0, K, T, r, sigma)
print(f"BSM closed-form: {bsm_price:.4f}")

# MC pricing
n_sims = 100_000
ST = simulate_gbm_terminal(S0, r, sigma, T, n_sims)
payoffs = np.maximum(ST - K, 0)
mc_price = np.exp(-r * T) * np.mean(payoffs)
mc_stderr = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_sims)

print(f"MC estimate:     {mc_price:.4f} +/- {mc_stderr:.4f} (std error)")
print(f"Difference:      {abs(mc_price - bsm_price):.4f}")


The MC estimate is close to BSM but not exact -- that's sampling noise, not a bug.
With 100K paths, the standard error is about $0.02-0.03. For production pricing
of exotics, desks typically use 500K to 2M paths.


## Convergence: How Many Paths Do You Need?

MC converges at rate $O(1/\sqrt{N})$. To cut error in half, you need 4x the paths.
Let's plot this convergence explicitly.


In [ ]:
path_counts = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000]
mc_estimates = []
mc_errors = []

for n in path_counts:
    ST_n = simulate_gbm_terminal(S0, r, sigma, T, n, seed=42)
    pay = np.maximum(ST_n - K, 0)
    est = np.exp(-r * T) * np.mean(pay)
    err = np.exp(-r * T) * np.std(pay) / np.sqrt(n)
    mc_estimates.append(est)
    mc_errors.append(err)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogx(path_counts, mc_estimates, 'o-', color='steelblue', lw=2)
axes[0].axhline(bsm_price, color='firebrick', ls='--', lw=1.5, label=f'BSM = {bsm_price:.4f}')
axes[0].fill_between(path_counts,
                     [e - 2*s for e, s in zip(mc_estimates, mc_errors)],
                     [e + 2*s for e, s in zip(mc_estimates, mc_errors)],
                     alpha=0.2, color='steelblue')
axes[0].set_xlabel('Number of Paths')
axes[0].set_ylabel('MC Price')
axes[0].set_title('MC Convergence to BSM')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].loglog(path_counts, mc_errors, 'o-', color='firebrick', lw=2, label='MC Std Error')
# Theoretical 1/sqrt(N) line
ref_N = np.array(path_counts, dtype=float)
ref_err = mc_errors[0] * np.sqrt(path_counts[0]) / np.sqrt(ref_N)
axes[1].loglog(path_counts, ref_err, '--', color='grey', label=r'$O(1/\sqrt{N})$')
axes[1].set_xlabel('Number of Paths')
axes[1].set_ylabel('Standard Error')
axes[1].set_title('Convergence Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Barrier Option: Down-and-Out Call

Now the fun part. A down-and-out call pays like a regular call, but it becomes
worthless ("knocks out") if the spot ever touches a barrier level during the life
of the option. This is path-dependent -- you need the full path, not just the
terminal value. No closed-form in BSM (well, there is one for continuous monitoring,
but real barriers are discrete, and the formula is ugly).

On the desk, these are bread-and-butter structured products. A client wants cheap
downside protection? Sell them a knock-out -- it's cheaper because they lose it if
things get really bad. The bank keeps the discontinuity risk.


In [ ]:
barrier = 90  # knock-out if S ever drops to 90

n_paths_barrier = 200_000
n_steps_barrier = 252  # daily monitoring

paths_barrier = simulate_gbm_paths(S0, r, sigma, T, n_steps_barrier,
                                    n_paths_barrier, seed=777)

# Check if barrier was breached on each path
min_along_path = np.min(paths_barrier, axis=0)  # minimum price for each path
survived = min_along_path > barrier

# Terminal payoff only for paths that survived
ST_barrier = paths_barrier[-1, :]
payoffs_barrier = np.where(survived, np.maximum(ST_barrier - K, 0), 0.0)

doc_price = np.exp(-r * T) * np.mean(payoffs_barrier)
doc_stderr = np.exp(-r * T) * np.std(payoffs_barrier) / np.sqrt(n_paths_barrier)
knockout_pct = 100 * (1 - np.mean(survived))

print(f"Down-and-Out Call (barrier={barrier}):")
print(f"  MC price:        {doc_price:.4f} +/- {doc_stderr:.4f}")
print(f"  Vanilla call:    {bsm_price:.4f}")
print(f"  Discount:        {(1 - doc_price / bsm_price) * 100:.1f}%")
print(f"  Knock-out rate:  {knockout_pct:.1f}%")


In [ ]:
# Visualize some paths -- green survived, red knocked out
fig, ax = plt.subplots(figsize=(12, 5))

# Plot a sample of each type
n_show = 30
survived_idx = np.where(survived)[0][:n_show]
knocked_idx = np.where(~survived)[0][:n_show]

for i in survived_idx:
    ax.plot(t_axis, paths_barrier[:, i], color='seagreen', alpha=0.3, lw=0.5)
for i in knocked_idx:
    ax.plot(t_axis, paths_barrier[:, i], color='firebrick', alpha=0.3, lw=0.5)

ax.axhline(barrier, color='firebrick', ls='--', lw=2, label=f'Barrier = {barrier}')
ax.axhline(K, color='grey', ls=':', lw=1, label=f'Strike = {K}')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Spot Price')
ax.set_title('Down-and-Out Call -- Path Visualization')
ax.legend()
plt.tight_layout()
plt.show()


## Variance Reduction (Brief Overview)

Plain MC converges slowly. Here are two techniques that can cut your simulation
time dramatically. I'm not implementing them fully here -- each deserves its own
deep dive -- but you should know they exist.

### Antithetic Variates
For every random draw $Z$, also compute the path with $-Z$. This ensures your
sample is perfectly symmetric, killing off the linear component of the error.
Implementation is trivial and roughly halves the variance for free.


In [ ]:
# Antithetic variates -- quick demo
rng = np.random.default_rng(42)
Z = rng.standard_normal(50_000)
Z_anti = np.concatenate([Z, -Z])  # 100K total, perfectly symmetric

ST_anti = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_anti)
payoffs_anti = np.maximum(ST_anti - K, 0)
mc_anti = np.exp(-r * T) * np.mean(payoffs_anti)
mc_anti_err = np.exp(-r * T) * np.std(payoffs_anti) / np.sqrt(len(Z_anti))

print(f"Antithetic MC (100K paths): {mc_anti:.4f} +/- {mc_anti_err:.4f}")
print(f"Plain MC (100K paths):      {mc_price:.4f} +/- {mc_stderr:.4f}")
print(f"BSM:                        {bsm_price:.4f}")


### Control Variates
Use a correlated variable with a known expectation to reduce variance. For example,
the stock price itself: $E[S_T] = S_0 e^{rT}$ is known. If your MC estimate of
$E[S_T]$ is too high, your option price estimate is probably too high by a
correlated amount. Subtract the error.

In practice, using the stock as a control variate can reduce MC variance by 50-80%
for ATM options. For more complex products, you use the closest available
closed-form proxy as the control.


In [ ]:
# Control variate demo
ST_cv = simulate_gbm_terminal(S0, r, sigma, T, 100_000, seed=42)
payoffs_cv = np.maximum(ST_cv - K, 0)

# The control: E[ST] = S0 * exp(rT)
expected_ST = S0 * np.exp(r * T)
control = ST_cv - expected_ST  # zero expectation

# Optimal coefficient (minimize variance of adjusted estimator)
cov_matrix = np.cov(payoffs_cv, control)
beta_cv = cov_matrix[0, 1] / cov_matrix[1, 1]

adjusted_payoffs = payoffs_cv - beta_cv * control
mc_cv = np.exp(-r * T) * np.mean(adjusted_payoffs)
mc_cv_err = np.exp(-r * T) * np.std(adjusted_payoffs) / np.sqrt(100_000)

print(f"Control Variate MC: {mc_cv:.4f} +/- {mc_cv_err:.4f}")
print(f"Plain MC:           {mc_price:.4f} +/- {mc_stderr:.4f}")
print(f"BSM:                {bsm_price:.4f}")
print(f"Variance reduction: {(1 - mc_cv_err / mc_stderr) * 100:.1f}%")


## Practical Notes

- **Always validate MC against a known solution first.** If your MC can't match
  BSM for a European call, it won't give you reliable prices for exotics.

- **Use vectorized numpy, not Python loops over paths.** The speed difference is
  100x. I've seen juniors write a for-loop over 1M paths and wonder why it takes
  20 minutes.

- **Seed your RNG for reproducibility.** When debugging, you need to reproduce
  the same paths. In production, use different seeds for different risk scenarios.

- **Barrier options have a discretization bias.** With daily monitoring and a
  barrier at 90, some paths that would have hit 90 intraday will slip through
  your daily grid. The Brownian bridge correction fixes this -- look it up if
  you're pricing barriers seriously.

- **On the desk, MC is slow but trustworthy.** When the quants deliver a new
  closed-form approximation for some exotic, the first thing we do is benchmark
  it against MC with 10M paths. If it disagrees, the closed form is wrong.

---

This concludes the core modules of the derivatives toolkit. From here, you have
the building blocks to price any vanilla or path-dependent option, understand
the vol surface, and think about risk the way a desk does.

*Djellal Djouad -- CrossVol Research -- 2026*
